# Tabular Prior Plots

This notebook resolves the live config builders in `configs/transformer/transformer_config.py` and `configs/fla/fla_config.py`, then samples through the same `TabPFNPriorConfig` adapter used during training.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from configs.fla.fla_config import MODEL_SETTINGS as FLA_MODEL_SETTINGS
from configs.fla.fla_config import get_config as get_fla_config
from configs.transformer.transformer_config import get_config as get_transformer_config
from pfns.priors.tabpfn_prior_adapter import TabPFNPriorConfig

plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["axes.grid"] = True


## Resolve The Live Pretraining Setup

The current transformer `high` setup and the current FLA `high` setups all call the same shared tabular prior builder and the same batch-shape sampler. We assert that here so the sampling below stays attached to the actual training path.


In [ ]:
transformer_main_config = get_transformer_config(training_setup="high")
fla_main_configs = {
    model_type: get_fla_config(
        training_setup="high",
        model_type=model_type,
        sequence_mode="Comb_ST",
    )
    for model_type in sorted(FLA_MODEL_SETTINGS)
}

prior_config = transformer_main_config.priors[0]
batch_shape_config = transformer_main_config.batch_shape_sampler

assert isinstance(prior_config, TabPFNPriorConfig)
assert all(cfg.priors[0] == prior_config for cfg in fla_main_configs.values())
assert all(cfg.batch_shape_sampler == batch_shape_config for cfg in fla_main_configs.values())

def _strip_config_type(payload: dict) -> dict:
    return {key: value for key, value in payload.items() if key != "__config_type__"}

display(pd.DataFrame([_strip_config_type(prior_config.to_dict())], index=["shared_tabular_prior"]))
display(pd.DataFrame([_strip_config_type(batch_shape_config.to_dict())], index=["shared_batch_shape_sampler"]))
print("Verified shared setup across transformer and FLA model types:", ", ".join(sorted(fla_main_configs)))


## Sample Dataset-Level Prior Draws


In [ ]:
def rounded_unique_count(values: torch.Tensor, decimals: int = 6) -> int:
    finite_values = values[torch.isfinite(values)]
    if finite_values.numel() == 0:
        return 0
    scaled = torch.round(finite_values.to(torch.float64) * (10 ** decimals))
    return int(torch.unique(scaled).numel())


def sample_prior_statistics(main_config, *, num_datasets: int, epoch: int = 1, start_step: int = 0):
    get_batch = main_config.priors[0].create_get_batch_method()
    dataset_rows = []
    categorical_rows = []

    for dataset_index in range(num_datasets):
        step = start_step + dataset_index
        batch_shape = main_config.batch_shape_sampler.sample_batch_shape(epoch=epoch, step=step)
        batch = get_batch(
            batch_size=1,
            seq_len=batch_shape.seq_len,
            num_features=batch_shape.num_features,
            single_eval_pos=batch_shape.single_eval_pos,
            n_targets_per_input=1,
        )

        x = batch.x.detach().cpu()
        if x.ndim == 3:
            x = x[0]

        if batch.categorical_mask is None:
            categorical_mask = torch.zeros(batch_shape.num_features, dtype=torch.bool)
        else:
            categorical_mask = batch.categorical_mask.detach().cpu()
            if categorical_mask.ndim == 2:
                categorical_mask = categorical_mask[0]
            categorical_mask = categorical_mask.to(torch.bool)

        nan_entries = torch.isnan(x)
        nan_fraction = float(nan_entries.to(torch.float32).mean().item())
        feature_has_nan = nan_entries.any(dim=0)

        categorical_feature_indices = torch.nonzero(categorical_mask, as_tuple=False).flatten().tolist()
        distinct_value_counts = [
            rounded_unique_count(x[:, feature_index])
            for feature_index in categorical_feature_indices
        ]

        dataset_rows.append(
            {
                "dataset_index": dataset_index,
                "epoch": epoch,
                "step": step,
                "seq_len": batch_shape.seq_len,
                "single_eval_pos": batch_shape.single_eval_pos,
                "num_features": batch_shape.num_features,
                "num_categorical_features": len(categorical_feature_indices),
                "categorical_feature_fraction": (
                    len(categorical_feature_indices) / batch_shape.num_features
                ),
                "num_nan_entries": int(nan_entries.sum().item()),
                "nan_fraction": nan_fraction,
                "num_features_with_nan": int(feature_has_nan.sum().item()),
            }
        )

        for feature_index, distinct_values in zip(categorical_feature_indices, distinct_value_counts):
            categorical_rows.append(
                {
                    "dataset_index": dataset_index,
                    "step": step,
                    "seq_len": batch_shape.seq_len,
                    "single_eval_pos": batch_shape.single_eval_pos,
                    "num_features": batch_shape.num_features,
                    "feature_index": feature_index,
                    "distinct_values": distinct_values,
                }
            )

    return pd.DataFrame(dataset_rows), pd.DataFrame(categorical_rows)


In [ ]:
NUM_DATASETS = 4096
EPOCH = 1
START_STEP = 0

dataset_stats, categorical_stats = sample_prior_statistics(
    transformer_main_config,
    num_datasets=NUM_DATASETS,
    epoch=EPOCH,
    start_step=START_STEP,
)

print(f"Sampled {len(dataset_stats)} datasets.")
print(f"Observed {len(categorical_stats)} categorical features across those datasets.")
display(dataset_stats.head())
display(categorical_stats.head())
display(dataset_stats[["num_features", "num_categorical_features", "categorical_feature_fraction", "num_nan_entries", "nan_fraction", "num_features_with_nan"]].describe())
if not categorical_stats.empty:
    display(categorical_stats[["distinct_values"]].describe())


In [ ]:
if categorical_stats.empty:
    raise RuntimeError("No categorical features were observed in the sampled prior datasets.")

def plot_discrete_distribution(ax, counts, *, color, title, xlabel, ylabel):
    positions = np.arange(len(counts))
    ax.bar(positions, counts.values, color=color)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if len(counts) <= 12:
        tick_positions = positions
    else:
        tick_positions = np.unique(np.linspace(0, len(counts) - 1, num=12, dtype=int))
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([str(counts.index[i]) for i in tick_positions], rotation=45, ha="right")

categorical_feature_counts = dataset_stats["num_categorical_features"].value_counts().sort_index()
distinct_value_counts = categorical_stats["distinct_values"].value_counts().sort_index()
mean_categorical_by_num_features = (
    dataset_stats.groupby("num_features", as_index=True)["num_categorical_features"]
    .mean()
    .sort_index()
)

fig, axes = plt.subplots(3, 2, figsize=(14, 15))

plot_discrete_distribution(
    axes[0, 0],
    categorical_feature_counts,
    color="#0b6e4f",
    title="Categorical Features Per Dataset",
    xlabel="Number of categorical features",
    ylabel="Dataset count",
)

plot_discrete_distribution(
    axes[0, 1],
    distinct_value_counts,
    color="#c97b22",
    title="Distinct Values Per Categorical Feature",
    xlabel="Distinct values in sampled feature",
    ylabel="Categorical feature count",
)

axes[1, 0].plot(
    mean_categorical_by_num_features.index,
    mean_categorical_by_num_features.values,
    marker="o",
    linewidth=2,
    color="#355070",
)
axes[1, 0].set_title("Mean Categorical Features By Total Features")
axes[1, 0].set_xlabel("Sampled total number of features")
axes[1, 0].set_ylabel("Mean categorical features")

axes[1, 1].hist(dataset_stats["categorical_feature_fraction"], bins=20, color="#6d597a", edgecolor="white")
axes[1, 1].set_title("Categorical Feature Fraction Per Dataset")
axes[1, 1].set_xlabel("Fraction of features that are categorical")
axes[1, 1].set_ylabel("Dataset count")

plot_discrete_distribution(
    axes[2, 0],
    categorical_feature_counts,
    color="#0b6e4f",
    title="Categorical Features Per Dataset (Log Y)",
    xlabel="Number of categorical features",
    ylabel="Dataset count",
)
axes[2, 0].set_yscale("log")

plot_discrete_distribution(
    axes[2, 1],
    distinct_value_counts,
    color="#c97b22",
    title="Distinct Values Per Categorical Feature (Log Y)",
    xlabel="Distinct values in sampled feature",
    ylabel="Categorical feature count",
)
axes[2, 1].set_yscale("log")

fig.suptitle("Categorical Structure In The Shared Transformer/FLA Pretraining Prior", y=1.01, fontsize=14)
fig.tight_layout()


## NaN Structure

In [ ]:
nan_distribution = (
    dataset_stats["num_nan_entries"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("dataset_probability")
    .reset_index()
)
nan_distribution.columns = ["num_nan_entries", "dataset_probability"]

nan_feature_distribution = (
    dataset_stats["num_features_with_nan"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("dataset_probability")
    .reset_index()
)
nan_feature_distribution.columns = ["num_features_with_nan", "dataset_probability"]

nan_entry_counts = dataset_stats["num_nan_entries"].value_counts().sort_index()
nan_feature_counts = dataset_stats["num_features_with_nan"].value_counts().sort_index()
datasets_with_any_nan = int((dataset_stats["num_nan_entries"] > 0).sum())
share_with_any_nan = datasets_with_any_nan / len(dataset_stats)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_discrete_distribution(
    axes[0, 0],
    nan_entry_counts,
    color="#b56576",
    title="NaN Entries Per Dataset",
    xlabel="Number of NaN entries",
    ylabel="Dataset count",
)
axes[0, 0].set_yscale("log")

axes[0, 1].hist(dataset_stats["nan_fraction"], bins=20, color="#e56b6f", edgecolor="white")
axes[0, 1].set_title("NaN Fraction Per Dataset")
axes[0, 1].set_xlabel("Fraction of entries that are NaN")
axes[0, 1].set_ylabel("Dataset count")
axes[0, 1].set_yscale("log")

plot_discrete_distribution(
    axes[1, 0],
    nan_feature_counts,
    color="#8d99ae",
    title="Features With NaNs Per Dataset",
    xlabel="Number of features containing at least one NaN",
    ylabel="Dataset count",
)
axes[1, 0].set_yscale("log")

axes[1, 1].axis("off")
summary_lines = [
    f"Datasets sampled: {len(dataset_stats)}",
    f"Datasets with any NaN: {datasets_with_any_nan}",
    f"Share with any NaN: {share_with_any_nan:.3f}",
    f"Max NaN entries in one dataset: {int(dataset_stats['num_nan_entries'].max())}",
    f"Max features with NaNs in one dataset: {int(dataset_stats['num_features_with_nan'].max())}",
]
if datasets_with_any_nan == 0:
    summary_lines.append("")
    summary_lines.append("In this sample all datasets had zero NaNs.")
    summary_lines.append("That can happen because NaN probabilities are sampled stochastically in the prior.")
axes[1, 1].text(0.0, 1.0, "\n".join(summary_lines), va="top", ha="left", family="monospace")

fig.suptitle("NaN Structure In The Shared Transformer/FLA Pretraining Prior", y=1.02, fontsize=14)
fig.tight_layout()

display(nan_distribution)
display(nan_feature_distribution)
